In [2]:
import os
import time
import copy
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report

# 1. Hardware Initialization and Volatile Cache Scrubber
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Remote workspace absolute source directories
train_dir = '/kaggle/input/datasets/simranvolunesia/pest-dataset/pest/train'
val_dir = '/kaggle/input/datasets/simranvolunesia/pest-dataset/pest/test'

# Standard structural boundaries
img_size = 224 
batch_size = 32
epochs = 25
patience = 5

class EarlyStopping:
    def __init__(self, patience=5, delta=1e-4):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# Advanced spatial transformations to prevent ambient pattern memorization
train_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(train_dir, train_tf)
val_data = datasets.ImageFolder(val_dir, val_tf)
classes = train_data.classes
num_classes = len(classes)

# 2. Volumetric Calculations for Inverse Weighted Sampling
class_counts = np.bincount(train_data.targets)
class_weights = 1.0 / class_counts
sample_weights = np.array([class_weights[t] for t in train_data.targets])
sample_weights = torch.from_numpy(sample_weights).double()
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_data, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# 3. Model Specialization via Deep Transfer Learning
model = models.mobilenet_v3_large(weights='DEFAULT')
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler('cuda')
stopper = EarlyStopping(patience=patience)

# Academic metric mapping array trackers
history = {
    'train_acc': [], 'val_acc': [], 
    'train_loss': [], 'val_loss': [],
    'learning_rates': []
}
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

print(f"Starting optimized training loop for {num_classes} pest categories...")

for epoch in range(epochs):
    start_t = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    history['learning_rates'].append(current_lr)
    
    # Gradient tracking loop step
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        t_loss += loss.item() * imgs.size(0)
        _, p = torch.max(out, 1)
        t_total += lbls.size(0)
        t_correct += (p == lbls).sum().item()

    # Evaluation step
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
                loss = criterion(out, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, p = torch.max(out, 1)
            v_total += lbls.size(0)
            v_correct += (p == lbls).sum().item()

    epoch_loss = v_loss / v_total
    epoch_acc = v_correct / v_total
    
    history['train_acc'].append(t_correct / t_total)
    history['val_acc'].append(epoch_acc)
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(epoch_loss)
    
    scheduler.step()
    print(f"Epoch {epoch+1:02d} | LR: {current_lr:.6f} | Val Acc: {epoch_acc*100:.2f}% | Val Loss: {epoch_loss:.4f} | Time: {time.time()-start_t:.1f}s")

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())
    
    stopper(epoch_loss)
    if stopper.early_stop:
        print("Early stopping triggered successfully.")
        break

# Serialization of optimal parameter array matrices
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/final_mobilenet_pest.pth')

# --- EXTENDED ACADEMIC VISUALIZATION ENGINE ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'
out_path = '/kaggle/working/'

# Figure 1: Dataset Balancing Profile Plot (Justifies Sampler in Methodology)
plt.figure(figsize=(10, 4))
plt.bar(range(num_classes), class_counts, color='teal', alpha=0.7, label='Raw Instance Frequency')
plt.axhline(y=np.mean(class_counts), color='red', linestyle='--', label='Mean Volumetric Baseline')
plt.title('Multi-Class Dataset Volumetric Distribution and Target Imbalances', fontsize=12, pad=12)
plt.xlabel('Pest Classification Target Index')
plt.ylabel('Available Frame Counts')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_class_distribution.png'))
plt.close()

# Figure 2: Three-Panel Optimization and Convergence Curves
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
ax1.plot(history['train_acc'], label='Training Vector', color='navy', linewidth=2, marker='o', markevery=2)
ax1.plot(history['val_acc'], label='Validation Vector', color='darkorange', linewidth=2, marker='s', markevery=2)
ax1.set_title('Categorical Classification Accuracy Path', fontsize=11)
ax1.set_xlabel('Optimization Epochs')
ax1.set_ylabel('Accuracy Ratio')
ax1.legend()

ax2.plot(history['train_loss'], label='Training Loss', color='crimson', linewidth=2, marker='o', markevery=2)
ax2.plot(history['val_loss'], label='Validation Loss', color='forestgreen', linewidth=2, marker='s', markevery=2)
ax2.set_title('Cross Entropy Objective Function Loss Curve', fontsize=11)
ax2.set_xlabel('Optimization Epochs')
ax2.set_ylabel('Objective Loss Value')
ax2.legend()

ax3.plot(history['learning_rates'], label='Learning Rate Decay Path', color='purple', linewidth=2, linestyle='-.')
ax3.set_title('Cosine Annealing Scheduler Path Profile', fontsize=11)
ax3.set_xlabel('Optimization Epochs')
ax3.set_ylabel('Calculated Step Boundaries')
ax3.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_curves.png'))
plt.close()

# Compute final validation indicators and measure real-time latency spread
all_preds, all_lbls = [], []
inference_latencies = []
model.eval()
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device)
        start_latency = time.time()
        out = model(imgs)
        _, p = torch.max(out, 1)
        latency_per_sample = (time.time() - start_latency) / imgs.size(0)
        inference_latencies.append(latency_per_sample)
        all_preds.extend(p.cpu().numpy())
        all_lbls.extend(lbls.numpy())

all_preds = np.array(all_preds)
all_lbls = np.array(all_lbls)

# Figure 3: Macro Confusion Matrix Map Overview
cm = confusion_matrix(all_lbls, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
plt.title('Multi-Tiered Topological Framework Verification Matrix Overview', fontsize=12, pad=12)
plt.xlabel('Predicted Pest Target Classes Index')
plt.ylabel('Ground Truth Pest Target Classes Index')
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_cm.png'))
plt.close()

# Figure 4: Multi-Class Statistical Parameter Grid Profile
report = classification_report(all_lbls, all_preds, target_names=classes, output_dict=True)
report_df = pd.DataFrame(report).iloc[:-1, :num_classes].T
plt.figure(figsize=(11, 9))
sns.heatmap(report_df.sample(n=min(30, num_classes), random_state=42), annot=True, fmt='.3f', cmap='YlGnBu', cbar=True)
plt.title('Fine-Grained Statistical Performance Parameter Grid Sample Profile', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_metrics_heatmap.png'))
plt.close()

# Figure 5: Box Plot Dispersion Limits (Proves Cross-Class Stability)
precision_scores = [report_df.loc[cls, 'precision'] for cls in report_df.index]
recall_scores = [report_df.loc[cls, 'recall'] for cls in report_df.index]
f1_scores = [report_df.loc[cls, 'f1-score'] for cls in report_df.index]

plt.figure(figsize=(7, 4.5))
plt.boxplot([precision_scores, recall_scores, f1_scores], tick_labels=['Precision Spread', 'Recall Spread', 'F1-Score Spread'])
plt.title('Axiomatic Statistical Reliability Range Analysis Across Pest Layouts', fontsize=12, pad=12)
plt.ylabel('Evaluated Boundary Metric Ratio')
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_metrics_boxplot.png'))
plt.close()

# Figure 6: Edge Node Execution Pass Duration Spread Histogram
plt.figure(figsize=(9, 4))
plt.hist(np.array(inference_latencies) * 1000, bins=20, color='darkslategray', edgecolor='black', alpha=0.85)
plt.axvline(x=np.mean(inference_latencies)*1000, color='red', linestyle='--', label=f'Mean Response: {np.mean(inference_latencies)*1000:.2f} ms')
plt.title('Embedded Hardware Localized Pass Duration Distribution Spread', fontsize=12, pad=12)
plt.xlabel('Single-Frame Forward Pass Duration (Milliseconds)')
plt.ylabel('Recorded Frame Density Frequencies')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'mobilenet_latency_profile.png'))
plt.close()

print("All advanced academic research artifacts generated and verified inside /kaggle/working/ folder.")

Starting optimized training loop for 9 pest categories...
Epoch 01 | LR: 0.001000 | Val Acc: 74.67% | Val Loss: 1.2576 | Time: 15.7s
Epoch 02 | LR: 0.000996 | Val Acc: 85.78% | Val Loss: 0.5542 | Time: 15.4s
Epoch 03 | LR: 0.000984 | Val Acc: 89.78% | Val Loss: 0.4497 | Time: 16.3s
Epoch 04 | LR: 0.000965 | Val Acc: 91.11% | Val Loss: 0.3694 | Time: 16.1s
Epoch 05 | LR: 0.000938 | Val Acc: 94.00% | Val Loss: 0.2783 | Time: 16.0s
Epoch 06 | LR: 0.000905 | Val Acc: 92.67% | Val Loss: 0.2989 | Time: 15.6s
Epoch 07 | LR: 0.000864 | Val Acc: 97.33% | Val Loss: 0.1591 | Time: 16.2s
Epoch 08 | LR: 0.000819 | Val Acc: 95.11% | Val Loss: 0.2078 | Time: 15.7s
Epoch 09 | LR: 0.000768 | Val Acc: 96.22% | Val Loss: 0.1660 | Time: 15.6s
Epoch 10 | LR: 0.000713 | Val Acc: 95.56% | Val Loss: 0.1940 | Time: 15.7s
Epoch 11 | LR: 0.000655 | Val Acc: 94.67% | Val Loss: 0.2334 | Time: 15.5s
Epoch 12 | LR: 0.000594 | Val Acc: 96.89% | Val Loss: 0.1707 | Time: 15.8s
Early stopping triggered successfully.
All